In [ ]:
import os
from typing import Iterable, Dict
import tensorflow as tf
import kerasncp as kncp
from kerasncp.tf import LTCCell, WiredCfcCell
from tensorflow import keras
import numpy as np
from matplotlib.image import imread
from tqdm import tqdm
from PIL import Image
import pandas as pd
import time
import matplotlib.pyplot as plt
import copy
import json
from dataclasses import dataclass, field, asdict
from typing import Tuple, Dict, Optional, List, Iterable, Union

from tensorflow.python.keras.layers import Conv2D, Dense
from tensorflow.python.keras.models import Functional

from keras_models import generate_ncp_model

In [ ]:
def generate_hidden_list(model: Functional, return_numpy: bool = True):
    """
    Generates a list of tensors that are used as the hidden state for the argument model when it is used in single-step
    mode. The batch dimension (0th dimension) is assumed to be 1 and any other dimensions (seq len dimensions) are
    assumed to be 0

    :param return_numpy: Whether to return output as numpy array. If false, returns as keras tensor
    :param model: Single step functional model to infer hidden states for
    :return: list of hidden states with 0 as value
    """
    constructor = np.zeros if return_numpy else tf.zeros
    hiddens = []
    if len(model.input_shape)==1:
        lool = model.input_shape[0][1:]
    else:
        # UPDATED CODE HERE input_shape[2:] -> input_shape[1:]
        lool = model.input_shape[1:]

    for input_shape in lool:  # ignore 1st output, as is this control output
        hidden = []
        for i, shape in enumerate(input_shape):
            if shape is None:
                if i == 0:  # batch dim
                    hidden.append(1)
                    continue
                elif i == 1:  # seq len dim
                    hidden.append(0)
                    continue
                else:
                    print("Unable to infer hidden state shape. Leaving as none")
            hidden.append(shape)
        hiddens.append(constructor(hidden))
    return hiddens

In [ ]:
models = ["../saved_models/retrain_difftraj_wscheduler0.85_seed22222_lr0.001_trainloss0.00016_valloss0.13141_diffcoreset900.h5",
          "../saved_models/retrain_150traj_wscheduler0.85_seed22222_lr0.001_trainloss0.00035_valloss0.00019_coreset900.h5"]
output_dirs = ["SINGLE_STEP_RETRAINED_DIFF_CORESET900", "SINGLE_STEP_RETRAINED_CORESET900"]

IMAGE_SHAPE = (144, 256, 3)
IMAGE_SHAPE_CV = (IMAGE_SHAPE[1], IMAGE_SHAPE[0])

DEFAULT_NCP_SEED = 22222

batch_size = None
seq_len = 64
augmentation_params = None
no_norm_layer = False
single_step = True

train_root = "../../fly_to_target_dataset/original_dataset"
# test_root = "../fly_to_target_dataset/test_data"

file_ending = 'png'

In [ ]:
def save_predictions(single_step_model, root, output_directory, file_type):
    train_inference_time = []
    test_inference_time = []
    
    hiddens = generate_hidden_list(model= single_step_model, return_numpy=True)
    print("hiddens shape: ", hiddens[0].shape)

    for directory in range(len(os.listdir(root))):
        predictions = []
        times = []
        directory_path = f"{root}/{directory + 1}"
        print("Processing directory : ", directory_path)
        n_images = [path for path in os.listdir(directory_path) if file_ending in path]
        print("Number of images :", len(n_images))
        for i in range(len(n_images)):
            current_image_path = f"{directory_path}/Image{i + 1}.png"
            current_img = Image.open(current_image_path).resize(IMAGE_SHAPE_CV)
            current_img_array = np.array(current_img)
            im_network = np.expand_dims(current_img_array, 0)
            start = time.time()
            output = single_step_model.predict([im_network, *hiddens])
            end= time.time()
            times.append(end - start)
            # print(end - start)
            predictions.append(list(output[0][0].tolist()))
            hiddens = output[1:]
        print("Avg Time: ", sum(times) / len(times))
        train_inference_time.append(sum(times) / len(times))
        # print(len(predictions))
        # print(predictions)
        columns = ['vx', 'vy', 'vz', 'omega_z']
        predictions_df = pd.DataFrame(predictions, columns=columns)

        # Save the DataFrame to a CSV file
        
        predictions_df.to_csv(f"{output_directory}/{file_type}_data{directory + 1}.csv", index=False)
        print("Predictions saved to file: ", directory + 1)



In [ ]:
def save_errors(root, output_directory, file_type):
    
    for directory in range(len(os.listdir(root))):
        csv_file_name = f'{root}/{directory + 1}/data_out.csv'
        labels = np.genfromtxt(csv_file_name, delimiter=',', skip_header=1, dtype=np.float32)

        preds_file_name = f'{output_directory}/{file_type}_data{directory + 1}.csv'
        pred_vals = np.genfromtxt(preds_file_name, delimiter=',', skip_header=1, dtype=np.float32)[:len(labels)]

        print(len(labels), len(pred_vals))
        error = labels - pred_vals
        
        columns = ['vx', 'vy', 'vz', 'omega_z']
        error_df = pd.DataFrame(error, columns=columns)

        # Save the DataFrame to a CSV file
        error_df.to_csv(f'{output_directory}/errors_{file_type}_data{directory + 1}.csv', index=False)
        print("Errors saved to file: ", directory + 1)

In [ ]:
!export TF_CPP_MIN_LOG_LEVEL=2

In [ ]:
!pwd

In [ ]:
for i in range (len(models)):
    MODEL_FILE = models[i]
    output_directory = output_dirs[i]
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
        print("directory created", output_directory)

    single_step_model = generate_ncp_model(seq_len, IMAGE_SHAPE, augmentation_params, batch_size, DEFAULT_NCP_SEED, single_step, no_norm_layer)

    with tf.device('/cpu:0'):
        single_step_model.load_weights(MODEL_FILE)
    print("Model loaded")


    file_type = "train"
    save_predictions(single_step_model, train_root, output_directory, file_type)
    save_errors(train_root, output_directory, file_type)



In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def plot_pred(output_dirs, root):
    # Iterate over each directory of model outputs
    for output_directory in output_dirs:
        for directory in range(len(os.listdir(root))):
            # Load the original data
            labels = f"{root}/{directory + 1}/data_out.csv"
            original_data = pd.read_csv(labels)

            # Load the predicted data from the model
            preds_file_name = f'{output_directory}/train_data{directory + 1}.csv'
            pred_vals = pd.read_csv(preds_file_name)

            # Create subplots for vx, vy, vz, and omega_z in a single figure
            fig, axs = plt.subplots(4, 1, figsize=(10, 12))
            fig.suptitle(f"Directory {directory + 1} - Model {output_directory}")

            # Plot vx
            axs[0].plot(original_data['vx'], label='Original vx', color='blue')
            axs[0].plot(pred_vals['vx'], label='Predicted vx', color='red')
            axs[0].set_title('vx')
            axs[0].legend()

            # Plot vy
            axs[1].plot(original_data['vy'], label='Original vy', color='blue')
            axs[1].plot(pred_vals['vy'], label='Predicted vy', color='red')
            axs[1].set_title('vy')
            axs[1].legend()

            # Plot vz
            axs[2].plot(original_data['vz'], label='Original vz', color='blue')
            axs[2].plot(pred_vals['vz'], label='Predicted vz', color='red')
            axs[2].set_title('vz')
            axs[2].legend()

            # Plot omega_z
            axs[3].plot(original_data['omega_z'], label='Original omega_z', color='blue')
            axs[3].plot(pred_vals['omega_z'], label='Predicted omega_z', color='red')
            axs[3].set_title('omega_z')
            axs[3].legend()

            # Adjust layout and show the plots
            plt.tight_layout(rect=[0, 0, 1, 0.96])
            plt.show()

plot_pred(output_dirs, train_root)

In [ ]:
    
# for directory in range(len(os.listdir(test_root))):
#     predictions = []
#     times = []
#     directory_path = f"{test_root}/{directory + 1}"
#     print("Processing directory : ", directory_path)
#     n_images = [path for path in os.listdir(directory_path) if file_ending in path]
#     print("Number of images :", len(n_images))
#     for i in range(len(n_images)):
#         current_image_path = f"{directory_path}/Image{i + 1}.png"
#         current_img = Image.open(current_image_path).resize(IMAGE_SHAPE_CV)
#         current_img_array = np.array(current_img)
#         im_network = np.expand_dims(current_img_array, 0)
#         start = time.time()
#         output = single_step_model.predict([im_network, *hiddens])
#         end= time.time()
#         times.append(end - start)
#         # print(end - start)
#         predictions.append(list(output[0][0].tolist()))
#         hiddens = output[1:]
#     print("Avg Time: ", sum(times) / len(times))
#     test_inference_time.append(sum(times) / len(times))
#     # print(len(predictions))
#     # print(predictions)
#     predictions_df = pd.DataFrame(predictions)


#     # Save the DataFrame to a CSV file
#     predictions_df.to_csv(f"{output_directory}/predictions_test_data{directory + 1}.csv", index=False)
#     print("Predictions saved to file: ", directory + 1)

# for directory in range(len(os.listdir(test_root))):
#     print("Processing directory : ", directory + 1)
#     csv_file_name = f'{test_root}/{directory + 1}/data_out.csv'
#     labels = np.genfromtxt(csv_file_name, delimiter=',', skip_header=1, dtype=np.float32)

#     preds_file_name = f'{output_directory}/predictions_test_data{directory + 1}.csv'
#     # pred_vals = np.genfromtxt(preds_file_name, delimiter=',', skip_header=1, dtype=np.float32)[:len(labels)]
#     pred_vals = pd.read_csv(preds_file_name)
#     print(len(labels), len(pred_vals))
    
#     error = labels - pred_vals
    
#     error_df = pd.DataFrame(error)

#     # # Save the DataFrame to a CSV file
#     error_df.to_csv(f'{output_directory}/errors_test_data{directory + 1}.csv', index=False)
#     print("Errors saved to file: ", directory + 1)